# 04 - Churn Classification

Compare churn classifiers with and without PCA, then identify the best model.

In [ ]:
from pathlib import Path
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

TRAIN_TEST_DIR = Path('../data/train_test')
REPORTS_DIR = Path('../reports')
MODEL_DIR = Path('../models')

In [ ]:
def load_split(use_pca=False):
    suffix = '_pca_80' if use_pca else '_processed'
    X_train = pd.read_csv(TRAIN_TEST_DIR / f'X_train{suffix}.csv')
    X_test = pd.read_csv(TRAIN_TEST_DIR / f'X_test{suffix}.csv')
    y_train = pd.read_csv(TRAIN_TEST_DIR / 'y_train.csv').squeeze('columns')
    y_test = pd.read_csv(TRAIN_TEST_DIR / 'y_test.csv').squeeze('columns')
    return X_train, X_test, y_train, y_test

In [ ]:
experiments = [
    ('Logistic Regression sans PCA', LogisticRegression(max_iter=1000, class_weight='balanced'), False),
    ('Random Forest sans PCA', RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced'), False),
    ('Logistic Regression avec PCA', LogisticRegression(max_iter=1000, class_weight='balanced'), True),
    ('Random Forest avec PCA', RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced'), True),
]

results = []
for name, model, use_pca in experiments:
    X_train, X_test, y_train, y_test = load_split(use_pca=use_pca)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_proba),
    })

pd.DataFrame(results).sort_values('F1-score', ascending=False)

In [ ]:
pd.read_csv(REPORTS_DIR / 'classification_results.csv').sort_values('F1-score', ascending=False)

In [ ]:
metadata = joblib.load(MODEL_DIR / 'best_churn_model_metadata.joblib')
best_model = joblib.load(MODEL_DIR / 'best_churn_model.joblib')
metadata, best_model